# 🤖 Week 1 — LLM Fundamentals, Real Behaviours & First Chatbot
### A beginner's guide with live experiments

---

**By the end of this notebook you will understand:**
- What a token is, why it matters, and how to estimate real costs
- How an LLM predicts the next word (and why it sometimes gets it wrong)
- What system prompt and user prompt are — and how to control output format
- How to talk to GPT-4o-mini AND Claude Haiku in the same notebook
- Why models give confident wrong answers (hallucination) — 3 patterns with ground truth
- How to automatically detect unreliable answers with self-consistency
- How to fix stale answers using Tavily live search
- How to build a persistent multi-turn chatbot

**No prior AI experience needed. Every cell is explained before you run it.**

---

### 📦 Libraries used
| Library | What it does |
|---|---|
| `openai` | Talks to GPT models |
| `anthropic` | Talks to Claude models |
| `tiktoken` | Counts tokens (OpenAI's tokeniser) |
| `tavily-python` | Live web search - gives models fresh facts |
| `python-dotenv` | Loads API keys from a `.env` file |
| `requests` | HTTP calls (Wikipedia ground truth) |

---
### Agenda
1. Setup — install, API keys, shared utilities
2. Tokens & cost — what your budget actually buys
3. Temperature — the determinism dial
4. System prompt vs User prompt — shaping the model
5. Two models, same question — GPT vs Claude
6. Hallucination anatomy — 3 patterns with ground truth verification
7. Self-consistency — detecting unreliable answers automatically
8. Fix stale answers with Tavily live search
9. Interactive prompt lab — exercises
10. Chatbot — persistent multi-turn conversation with history


---
## ⚙️ Cell 0 — Install & Setup
Run this first. It installs everything and loads your API keys.

In [ ]:
# Install all libraries we need (pydantic required by shared/utils.py)
!pip install openai anthropic tiktoken tavily-python python-dotenv requests pydantic --quiet

In [ ]:
import sys, os, json, math, textwrap, requests
from dotenv import load_dotenv
from IPython.display import display, Markdown
import tiktoken
import math
# Load keys from .env file in the same folder
# Your .env file should look like this:
#   OPENAI_API_KEY=sk-...
#   ANTHROPIC_API_KEY=sk-ant-...
#   TAVILY_API_KEY=tvly-...
load_dotenv()

OPENAI_API_KEY    = os.getenv("OPENAI_API_KEY",    "")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")
TAVILY_API_KEY    = os.getenv("TAVILY_API_KEY",    "")

print("API Key Status:")
print(f"  OpenAI   : {'✅ loaded' if OPENAI_API_KEY    else '❌ missing — add to .env'}")
print(f"  Anthropic: {'✅ loaded' if ANTHROPIC_API_KEY else '❌ missing — add to .env'}")
print(f"  Tavily   : {'✅ loaded' if TAVILY_API_KEY    else '❌ missing — get free key at tavily.com'}")

from openai import OpenAI
import anthropic

oai_client    = OpenAI(api_key=OPENAI_API_KEY)
claude_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

print()
print("✅  Raw clients ready!")


### 🎓My note — mixed API style (on purpose)

**Week 1** keeps most calls on the **raw OpenAI / Anthropic SDKs** so you see exactly what each provider expects (`messages`, `system`, `temperature`, etc.).

**Shared utils** (`LLMClient`, `Guardrails`, `Tracer`) appear in a **few cells only** — a preview of what Week 2 standardises on.

| Style | Where | Why |
|---|---|---|
| Raw SDK | Sections 1–5, 7–9 | Learn the plumbing |
| `LLMClient` / `Guardrails` | Cost calc, self-consistency, session summary | See tracing & production helpers |

The session tracer at the end will **not** list every call — only those routed through `LLMClient`. That is expected for Week 1.


In [ ]:
# ── Load shared utilities (selective use in Week 1) ─────────────────────────
# Week 1 mostly uses raw oai_client / claude_client above.
# LLMClient + Guardrails appear in a few cells; Week 2 uses them throughout.

def _find_repo_root(*start_dirs):
    for start in start_dirs:
        if not start:
            continue
        root = os.path.abspath(start)
        while root and not os.path.isfile(os.path.join(root, "shared", "utils.py")):
            parent = os.path.dirname(root)
            if parent == root:
                root = None
                break
            root = parent
        if root:
            return root
    return None

_starts = [os.getcwd()]
_nb = globals().get("__vsc_ipynb_file__")  # set by VS Code / Cursor Jupyter
if _nb:
    _starts.insert(0, os.path.dirname(os.path.abspath(_nb)))

_repo_root = _find_repo_root(*_starts)
if _repo_root and _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

# Drop stale cached modules (e.g. after shared/utils.py was updated mid-session)
for _mod in list(sys.modules):
    if _mod == "shared" or _mod.startswith("shared."):
        del sys.modules[_mod]

try:
    from shared.utils import LLMClient, Tracer, Guardrails, banner, ask_gpt, ask_claude
    tracer = Tracer(session_id="w01")
    client = LLMClient(tracer=tracer)
    GPT   = "gpt-4o-mini"
    HAIKU = "claude-haiku-4-5"
    print("✅  Shared utils loaded — tracer session:", tracer.session_id)
except ImportError as e:
    print("⚠️  Could not import shared.utils:", e)
    if not _repo_root:
        print("   shared/utils.py not found — clone the repo or open the notebook from training_5weeks/")
    elif "pydantic" in str(e).lower():
        print("   Re-run the pip install cell above (needs pydantic).")
    else:
        print("   Use Kernel → Restart, then re-run the setup cells above.")
    tracer = None
    client = None
    GPT    = "gpt-4o-mini"
    HAIKU  = "claude-haiku-4-5"


---
## 🧩 Section 1 — Tokens & Cost: What Your Budget Actually Buys

An LLM does **not** read words. It reads **tokens** — chunks of text that can be:
- A whole word: `cat` → 1 token
- Part of a word: `unbelievable` → `un` + `believ` + `able` → 3 tokens
- A punctuation mark or space

**Why does this matter?**
- You pay per token (not per word)
- The model has a token limit (context window)
- Some tasks are hard for models *because* of tokenisation (like counting letters)
- Non-English text uses significantly more tokens per word


In [ ]:
# ── 1a. Token visualiser — what the model actually reads ─────────────────────
enc = tiktoken.encoding_for_model("gpt-4o")

def show_tokens(text):
    """Break text into tokens and display them visually."""
    tokens  = enc.encode(text)
    decoded = [enc.decode([t]) for t in tokens]
    print(f"📝 Input    : {text!r}")
    print(f"🔢 Tokens   : {decoded}")
    print(f"📊 Count    : {len(tokens)} tokens | {len(text)} characters | ratio={len(tokens)/max(len(text.split()),1):.1f} tok/word")
    print(f"💰 Cost est : ${len(tokens) * 0.00000015:.8f}  (at gpt-4o-mini input rate)")
    print()

print("=" * 65)
print("TOKEN VISUALISER")
print("=" * 65)
print()

show_tokens("Hello")
show_tokens("unbelievable")
show_tokens("supercalifragilisticexpialidocious")
show_tokens("P1 SLA breach RTO/RPO CMDB CI SLA")    # IT acronyms
show_tokens("नमस्ते — IT हेल्पडेस्क पर आपका स्वागत है")  #Hindi
show_tokens("Hallo — Willkommen beim IT-Helpdesk") # German
show_tokens("IT Helpdesk welcomes you")                # English equivalent
show_tokens("2024-11-07T14:30:00Z")                    # Timestamp                         # Used in next cell!

print("💡 Notice: Hindi uses ~3× more tokens than English for the same meaning.")
print("   Non-English users pay more AND hit context limits faster.")


In [ ]:
# ── The classic trick question ────────────────────────────────────────────────
# Ask the model to count letters. Because it sees TOKENS not characters,
# it will get some of these wrong.

print("=" * 60)
print("THE CHARACTER COUNTING TRAP")
print("=" * 60)
print()
print("Asking GPT-4o-mini to count letters...")
print("(It reads tokens, not characters — watch what happens)")
print()

questions = [
    ("Count the number of 'e's in 'electroencephalography'.",  "4"),
    ("How many characters are in the word 'queue'?", "5"),
    ("How many characters in the word 'Mississippi'?",     "11"),
]

for question, correct_answer in questions:
    response = oai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": question}],
        temperature=0
    )
    model_answer = response.choices[0].message.content.strip()

    # Check if the model got it right
    is_correct = correct_answer.lower() in model_answer.lower()
    icon = "✅ Correct" if is_correct else "❌ Wrong  "

    print(f"Q: {question}")
    print(f"   Model says : {model_answer[:80]}")
    print(f"   Truth      : {correct_answer}")
    print(f"   Result     : {icon}")
    print()

print("🔑 KEY LESSON: For counting, parsing, or formatting tasks : use Python code,")
print("   not an LLM. LLMs are pattern matchers, not calculators.")

In [ ]:
# 1c. Cost calculator — real production estimates  
#
if client is None:
    print("⚠️  LLMClient not loaded — showing manual formula instead.")
    print()
    print("  Formula: daily_cost = (input_tokens * 0.00015 + output_tokens * 0.00060) / 1000 * volume")
    print()
    scenarios = [
        ("Ticket classifier (ServiceNow auto-routing)",  200,  80, 500),
        ("Chatbot response (employee helpdesk)",         800, 300, 1000),
        ("Policy summariser (RAG over docs, Haiku)",    3000, 400, 200),
    ]
    print(f"{'Scenario':<45} {'Daily':>6} {'Daily $':>9} {'Monthly $':>10}")
    print("─" * 75)
    for name, inp, out, vol in scenarios:
        daily = (inp * 0.00015 + out * 0.00060) / 1000 * vol
        print(f"{name:<45} {vol:>6,} {daily:>9.4f} {daily*30:>10.2f}")
else:
    banner("Cost calculator — real production estimates")
    scenarios = [
        {"name": "Ticket classifier (ServiceNow auto-routing)",
         "avg_input_tokens": 200, "avg_output_tokens": 80,  "daily_volume": 500,  "model": GPT},
        {"name": "Chatbot response (employee helpdesk)",
         "avg_input_tokens": 800, "avg_output_tokens": 300, "daily_volume": 1000, "model": GPT},
        {"name": "Policy summariser (RAG over docs)",
         "avg_input_tokens": 3000,"avg_output_tokens": 400, "daily_volume": 200,  "model": HAIKU},
    ]
    print(f"{'Scenario':<45} {'Daily calls':>12} {'Daily cost $':>13} {'Monthly $':>10}")
    print("─" * 85)
    for s in scenarios:
        daily_cost = client.estimate_cost(
            prompt="x" * s["avg_input_tokens"],
            expected_output_tokens=s["avg_output_tokens"],
            model=s["model"]
        ) * s["daily_volume"]
        print(f"{s['name']:<45} {s['daily_volume']:>12,} {daily_cost:>13.4f} {daily_cost*30:>10.2f}")
    print()
    print("💡 The summariser (RAG) is cheapest per-call but most expensive in total")
    print("   because of large contexts. Context window management is a real cost lever.")


# Section 2 - Temperature, Top-K & Top-P - How LLMs Decide What to Say

## The Core Idea

Every time an LLM generates the next word, it does **not** pick randomly from all possible words.
It first calculates a probability score for every word in its vocabulary (~100,000 words),
then uses **three controls** to decide which of those candidates to actually sample from.

---

## 🌡️ Temperature — How Confident vs Creative

Temperature **scales** the probability distribution before any sampling happens.

| Temperature | Effect | Use when |
|-------------|--------|----------|
| `0.0` | Always picks the single highest-probability token. Fully deterministic. | Classification, routing, yes/no decisions |
| `0.3 – 0.5` | Slightly softened — still focused, minor variation across runs | Summarisation, structured reports |
| `0.7 – 0.9` | Noticeably varied — different phrasing each run | Drafting, brainstorming, creative writing |
| `1.0+` | Flat distribution — model treats unlikely tokens as nearly as valid as likely ones | Exploration only — unpredictable in production |

> **Analogy:** Temperature is the volume knob.  
> Low = the model whispers only its most confident answer.  
> High = the model starts treating wild guesses as equally worth saying.

---

## 🎯 Top-K — Hard Candidate Limit

Top-K **cuts** the vocabulary to only the K highest-probability tokens before sampling.

In [ ]:
# ── 2a. Visualise "next word prediction" with logprobs ───────────────────────
# OpenAI lets us see the top 5 tokens it considered at each step.
# This makes the prediction process VISIBLE.

def show_next_word_probabilities(prompt):
    response = oai_client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1,
        logprobs=True,
        top_logprobs=5,
        temperature=0.1
    )
    top5 = response.choices[0].logprobs.content[0].top_logprobs

    print(f"📝 Prompt: \"{prompt}\"")
    print(f"\n   Top {len(top5)} candidates for the NEXT word:")
    print(f"   {'Token':<20} {'Probability':>12} {'Bar'}")
    print(f"   {'─'*20} {'─'*12} {'─'*25}")
    for lp in top5:
        prob  = math.exp(lp.logprob) * 100
        bar   = "█" * int(prob / 2)
        token = repr(lp.token)
        print(f"   {token:<20} {prob:>11.1f}%  {bar}")
    print()

print("=" * 60)
print("NEXT WORD PROBABILITY VISUALISER")
print("=" * 60)
print()

show_next_word_probabilities("The capital of France is")    # Unambiguous
show_next_word_probabilities("Python is a programming")     # Slightly ambiguous
show_next_word_probabilities("The best way to fix a broken")  # Very ambiguous

print("🔑 KEY LESSON:")
print("   CERTAIN → one token has ~99% probability (Paris)")
print("   UNCERTAIN → probability spreads across many tokens")
print("   Hallucination = confident token picked despite no real knowledge to back it up.")

In [ ]:
# ── 2b. Temperature experiment — 3 tasks that show the difference ────────────
# KEY INSIGHT: temperature only shows visible variation on AMBIGUOUS or CREATIVE tasks.

print("=" * 65)
print("TEMPERATURE EXPERIMENT")
print("=" * 65)
print()

# Task A: AMBIGUOUS classification
print("── Task A: Ambiguous classification (network + software + access mix)")
AMBIGUOUS_TICKET = """
After the Windows 11 feature update, some staff cannot reach internal SharePoint
via office Wi-Fi, but it works on mobile data. The IT team is not sure if this is
DNS, a proxy config change from the update, or a SharePoint permission problem.
NOTE: This issue equally involves network routing, OS software, and user access tokens.
"""
SYS_CLASSIFY = ("Classify into ONE category: "
                "[Network, Hardware, Software, Access, Other]. Return only the category name.")

for label, temp in [("temp=0.0", 0.0), ("temp=1.5", 1.5)]:
    answers = []
    print(f"   {label} (15 runs):")
    for i in range(15):
        r = oai_client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role":"system","content":SYS_CLASSIFY},
                      {"role":"user","content":AMBIGUOUS_TICKET}],
            temperature=temp, max_tokens=20
        )
        ans = r.choices[0].message.content.strip()
        answers.append(ans)
        print(f"     Run {i+1}: {ans}")
    print(f"   → Unique: {set(answers)}")
    print()


In [ ]:
# Task B: CREATIVE tagline writing
print("── Task B: Creative task (tagline writing)")
CREATIVE_PROMPT = "Write one short, catchy tagline for an AI-powered IT helpdesk. Max 10 words."

for label, temp in [("temp=0.0", 0.0), ("temp=1.0", 1.0)]:
    seen = set()
    print(f"   {label} (5 runs):")
    for i in range(5):
        r = oai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": CREATIVE_PROMPT}],
            temperature=temp, max_tokens=30
        )
        ans = r.choices[0].message.content.strip()
        seen.add(ans)
        print(f"     Run {i+1}: {ans}")
    print(f"   → Unique answers: {len(seen)}/5")
    print()

print("🔑 RULE: temp=0 for decisions/routing. temp=0.7–1.0 for drafting/brainstorming.")
print("   In any automation based pipeline: ALWAYS use temp=0.")

In [ ]:
# ── 6c. Top-K / Top-P Sampling Demo ──────────────────────────────────────────
print("=" * 60)
print("6c. HOW LLMs DECIDE WHAT TO SAY — Top-K & Top-P")
print("=" * 60)

# ─────────────────────────────────────────────────────────────
# DEMO 1 — What tokens was the model choosing between?
# ─────────────────────────────────────────────────────────────
print("\n" + "─" * 60)
print("\n🔬 DEMO 1 — What was the model actually considering?")
print("   Completing: 'The most likely root cause of the outage was a ...'")
print("   (top 5 candidate next-words + their probabilities)\n")

r = oai_client.chat.completions.create(
    model        = "gpt-4o-mini",
    messages     = [{"role": "user",
                     "content": "Complete this sentence in 2 words only: "
                                "The most likely root cause of the outage was a"}],
    temperature  = 1.0,
    max_tokens   = 1,
    logprobs     = True,
    top_logprobs = 5,
)

first     = r.choices[0].logprobs.content[0]
chosen    = first.token
candidates = sorted(first.top_logprobs, key=lambda t: t.logprob, reverse=True)
max_pct   = math.exp(candidates[0].logprob) * 100

for t in candidates:
    pct     = math.exp(t.logprob) * 100
    bar     = "█" * int((pct / max_pct) * 30)
    marker  = "  ← model picked this" if t.token == chosen else ""
    print(f"  {t.token!r:<18} {bar:<30} {pct:>5.1f}%{marker}")

print("\n  💡 The model didn't 'know' the answer — it sampled from this list.")
print("  💡 Top-K=1 always picks the top bar. Top-K=3 samples from the top 3.")
print("  💡 This is why the same prompt can give different answers each run.")

# ─────────────────────────────────────────────────────────────
# DEMO 2 — top_p on a creative task (visible diversity)
# ─────────────────────────────────────────────────────────────
print("\n" + "─" * 60)
print("\n🎲 DEMO 3 — top_p: how wide is the sampling pool?")
print("   Prompt: 'What should an engineer do first during a P1 incident?'")
print("   (4 runs each — watch how much the opening word changes)\n")

CREATIVE_PROMPT = "What should an engineer do first during a P1 incident? Answer in one sentence."

for top_p, label in [(0.05, "Narrow pool — tiny vocab"), (0.95, "Wide pool  — full vocab")]:
    print(f"  top_p={top_p}  [{label}]")
    responses = []
    for i in range(4):
        r = oai_client.chat.completions.create(
            model       = "gpt-4o-mini",
            messages    = [{"role": "user", "content": CREATIVE_PROMPT}],
            temperature = 1.0,
            top_p       = top_p,
            max_tokens  = 15,
        )
        resp = r.choices[0].message.content.strip()
        responses.append(resp)
        print(f"    Run {i+1}: {resp}")

    # show first-word variety
    first_words = [r.split()[0].lower().strip(".,") for r in responses]
    unique_fw   = set(first_words)
    print(f"    Opening words: {first_words}  →  {len(unique_fw)} unique\n")

print("  💡 top_p=0.05  → opening word is almost always 'The' or 'Immediately'")
print("  💡 top_p=0.95  → opening word varies: 'Immediately', 'First', 'Alert', 'Notify'...")
print("  💡 For incident ROUTING use low top_p — you want the same answer every time")
print("  💡 For incident REPORTS use high top_p — you want natural, non-repetitive language")


# ─────────────────────────────────────────────────────────────
print("\n" + "─" * 60)
print("\n🔑 ONE-LINE SUMMARY\n")
print("   temperature → how peaked vs flat the distribution is")
print("   top_k       → keep only the K most likely tokens")
print("   top_p       → keep tokens until their probs sum to P")
print()
print("   For a bank AI:")
print("   • Routing decisions  → temp=0,   top_p=0.1   (deterministic)")
print("   • Writing summaries  → temp=0.3, top_p=0.85  (natural)")
print("   • Brainstorming      → temp=0.9, top_p=0.95  (creative)")

---
## 💬 Section 3 — System Prompt vs User Prompt

Every LLM conversation has two layers:

```
┌─────────────────────────────────────────────┐
│  SYSTEM PROMPT  (the instructions)           │
│  → Set by the developer , Owner         │
│  → Defines role, tone, rules, format         │
│  → User usually cannot see this              │
├─────────────────────────────────────────────┤
│  USER PROMPT  (the request)                  │
│  → Typed by the user                         │
│  → The actual question or task               │
└─────────────────────────────────────────────┘
```

**The system prompt is like a job description given to a new employee before they start.**
The user prompt is what the customer then asks that employee.


In [ ]:
# ── 3a. Same question, 3 different system prompts ────────────────────────────

USER_QUESTION = "My laptop is running very slow today."

print("=" * 60)
print("SYSTEM PROMPT SHAPES THE ANSWER")
print(f"User question: \"{USER_QUESTION}\"")
print("=" * 60)
print()

personas = [
    ("Friendly IT Helpdesk",
     "You are a friendly IT helpdesk assistant. Be warm, empathetic, and give 2-3 practical steps. Use simple language."),
    ("Senior Security Engineer",
     "You are a cybersecurity expert. Always consider security implications first. Be technical and precise."),
    ("Cost-Focused IT Manager",
     "You are an IT manager focused on cost efficiency. Prioritise free solutions and self-service before escalation. Be brief."),
]

for name, system_prompt in personas:
    response = ask_gpt(system_prompt, USER_QUESTION)
    print(f"── 🎭 Persona: {name} ──")
    print(f"   System: \"{system_prompt}...\"")
    print()
    for line in response.splitlines():
        print(f"   {line}")
    print()

print("🔑 KEY LESSON: The system prompt defines who the model IS, not just what to say.")

In [ ]:
# ── 3b. System prompt controls OUTPUT FORMAT ─────────────────────────────────

print("=" * 60)
print("SYSTEM PROMPT CONTROLS OUTPUT FORMAT")
print("=" * 60)
print()

incident = """
At 2:30 PM the entire finance floor lost access to SAP.
About 80 users affected. Root cause was an expired SSL certificate
on the load balancer. Fixed by 3:15 PM. Downtime: 45 minutes.
"""

format_prompts = [
    ("Bullet Points",
     "Summarise IT incidents as exactly 3 bullet points. Start each with •"),
    ("One Sentence",
     "Summarise IT incidents in exactly ONE sentence. Max 25 words."),
    ("JSON",
     'Summarise IT incidents as JSON: {"impact": str, "cause": str, "resolution": str, "downtime_mins": int}. Return only valid JSON.'),
]

for format_name, system_prompt in format_prompts:
    response = ask_gpt(system_prompt, incident, temperature=0)
    print(f"── Format: {format_name} ──")
    print(f"   System: \"{system_prompt[:60]}...\"")
    for line in response.splitlines():
        print(f"     {line}")
    print()


---
## 🤖 Section 4 — Two Models, Same Question

| Model | Company | Good at | Cost |
|---|---|---|---|
| `gpt-4o-mini` | OpenAI | Fast, cheap, reliable | Very low |
| `claude-haiku-4-5` | Anthropic | Careful, nuanced, refuses harmful requests | Very low |

They are trained differently, have different training data cutoffs,
and make different mistakes. **Never assume two models agreeing = correct.**


In [ ]:
# ── 4a. GPT vs Claude comparison ──────────────────────────────────────────────

system = "You are an IT helpdesk assistant. Be helpful and concise."

print("=" * 60)
print("GPT-4o-mini  vs  Claude Haiku")
print("=" * 60)
print()

comparison_tasks = [
    "In one sentence: what is the difference between an incident and a problem in ITIL?",
    "My VPN keeps disconnecting every 30 minutes. What are the 3 most likely causes?",
    "Should we store user passwords in our database in plain text? Answer in one word, then explain why.",
]

for task in comparison_tasks:
    gpt_ans    = ask_gpt(system, task, temperature=0)
    claude_ans = ask_claude(system, task, temperature=0)
    print(f"❓ Question: {task}")
    print()
    print(f"   🟢 GPT-4o-mini:")
    for line in gpt_ans.splitlines():
        print(f"      {line}")
    print()
    print(f"   🟣 Claude Haiku:")
    for line in claude_ans.splitlines():
        print(f"      {line}")
    print()
    print("   " + "─" * 55)
    print()


In [ ]:
# ── 4b. Where models disagree — and BOTH sound confident ─────────────────────
print("=" * 60)
print("DANGER ZONE: Models disagree — at least one is wrong")
print("=" * 60)
print()

factual_questions = [
    "What was ServiceNow's total revenue in fiscal year 2022-2023?",
    "Who is the current CEO of Wipro?",
]

system_factual = "Answer factually and precisely. Be direct. Keep it under 2 sentences."

def semantic_agree(question, ans_a, ans_b) -> tuple[str, str]:
    """Use a 3rd LLM call to judge semantic agreement — not string comparison."""
    judge_prompt = f"""
You are a neutral fact-checker.

Question: {question}
Answer A: {ans_a}
Answer B: {ans_b}

Do these two answers agree on the CORE factual claim?
Reply with exactly one of: AGREE / DISAGREE / PARTIAL
Then on the next line, one sentence explaining why.
"""
    verdict = ask_claude("You are a neutral judge. Be concise.", judge_prompt, temperature=0)
    lines   = verdict.strip().splitlines()
    label   = lines[0].strip().upper() if lines else "UNKNOWN"
    reason  = lines[1].strip() if len(lines) > 1 else ""
    return label, reason

for q in factual_questions:
    gpt_ans    = ask_gpt(system_factual, q, temperature=0)
    claude_ans = ask_claude(system_factual, q, temperature=0)

    label, reason = semantic_agree(q, gpt_ans, claude_ans)

    icon = {"AGREE": "✅", "DISAGREE": "⚠️ ", "PARTIAL": "🟡"}.get(label, "❓")

    print(f"❓ {q}")
    print(f"   🟢 GPT   : {gpt_ans[:120]}")
    print(f"   🟣 Claude: {claude_ans[:120]}")
    print(f"   {icon} Verdict: {label} — {reason}")
    print()

print("🔑 Model agreement does NOT equal truth.")
print("   Even when they agree — verify facts that matter with a primary source.")

---
## 🌀 Section 5 — Hallucination Anatomy

Hallucination is not a bug — it's a **structural consequence** of how LLMs work.

The model is always predicting the next most likely token. When it doesn't know something,
it predicts what a **plausible answer would look like** — and delivers that with full confidence.

| Type | Example |
|---|---|
| **Fabrication** | Inventing statistics, dates, people, or DOIs that don't exist |
| **Outdated fact** | Stating something that was true 2 years ago as current |
| **Conflation** | Mixing facts from two similar things (two CEOs, two products) |

We verify all three against **Wikipedia live data**.


In [ ]:
import requests
import re

# ── Wikipedia ground truth helper ────────────────────────────────────────────
def wiki(topic: str, chars: int = 400) -> str:
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{topic.replace(' ','_')}"
    r = requests.get(url, headers={"User-Agent": "TrainingDemo/1.0"}, timeout=5)
    if r.status_code == 200:
        return r.json().get("extract", "Not found")[:chars]
    return f"Wikipedia returned {r.status_code}"

In [ ]:
def wiki_fact(topic: str, question: str) -> str:
    """Extract only the sentence from Wikipedia that answers the question."""
    extract = wiki(topic)
    prompt  = f"""
From this Wikipedia text, extract ONLY the single sentence or short phrase 
that directly answers: "{question}"

Wikipedia text:
{extract}

Reply with just the relevant fact. Maximum 20 words. 
If not found, reply: "Not in Wikipedia extract."
"""
    result = ask_claude("You extract specific facts from text. Be extremely concise.", prompt, temperature=0)
    return result.strip()


# ── Hallucination Type 1: Fabricated specifics about real companies ───────────
print("=" * 60)
print("HALLUCINATION TYPE 1 — Fabricated Facts")
print("=" * 60)
print()

questions = [
    ("When was ServiceNow founded and by whom?",            "ServiceNow"),
    ("Who is the current CEO of Wipro?",                  "Wipro"),
    ("What was Wipro's revenue in their last fiscal year?", "Wipro"),
]

for q, wiki_topic in questions:
    gpt_ans    = ask_gpt("Answer in one sentence. Be direct and factual.", q, temperature=0)
    claude_ans = ask_claude("Answer in one sentence. Be direct and factual.", q, temperature=0)
    wk_fact    = wiki_fact(wiki_topic, q)

    print(f"❓ {q}")
    print(f"   🟢 GPT   : {gpt_ans.strip()}")
    print(f"   🟣 Claude: {claude_ans.strip()}")
    print(f"   📖 Wiki  : {wk_fact}")
    print()

print("─" * 60)
print("💡 GPT says ServiceNow founded 2004 — Wiki confirms 2003.")
print("   Claude gets it right. Both sound equally confident.")
print("=" * 60)

In [ ]:
# ── Hallucination Type 2: Invented academic citations ────────────────────────
print("=" * 60)
print("HALLUCINATION TYPE 2 — Fabricated Citations")
print("=" * 60)
print()

citation_prompt = """
Give me 3 academic papers on 'ROI of IT automation in enterprise'.
For each paper include:
- Author(s)
- Title
- Journal name
- Year
- DOI (format: 10.XXXX/XXXXX)

Return as a numbered list. Be specific with real-looking DOIs.
"""

response =ask_gpt("You are a research assistant.", citation_prompt, model="smart", temperature=0)
print("📚 Model citations:\n")
print(response)
print()

# ── Extract DOIs and build clickable verification links ──────────────────────
dois = re.findall(r'10\.\d{4,}/\S+', response)
dois = [d.rstrip('.,)') for d in dois]   # strip trailing punctuation

print("─" * 60)
print("🔍 LIVE VERIFICATION LINKS — click each to check if it exists:")
print()
for i, doi in enumerate(dois, 1):
    doi_url    = f"https://doi.org/{doi}"
    scholar_url = f"https://scholar.google.com/scholar?q={doi}"
    print(f"   Paper {i}:")
    print(f"   DOI URL    → {doi_url}")
    print(f"   Google Scholar → {scholar_url}")
    print()

print("⚠️  EXPECTED RESULT:")
print("   Most DOI links will 404 or resolve to a completely different paper.")
print("   The models fabricated plausible-looking citations — this is hallucination.")
print("   In professional / research settings this is dangerous.")
print("=" * 60)

In [ ]:
# ── Hallucination Type 3: Conflation + Self-knowledge inconsistency ──────────
print("=" * 60)
print("HALLUCINATION TYPE 3 — Conflation")
print("=" * 60)
print()

# These questions are designed to trigger real conflation errors:
# - Models often mix up PRINCE2 vs PMP process groups
# - Models often confuse ISO 27001 control counts across versions
# - Models often conflate TOGAF phases with Zachman columns

conflation_questions = [
    {
        "q": "How many processes are in PRINCE2 2017 edition?",
        "truth": "7 processes",
        "trap": "Models often say 8 (confusing with PMBOK process groups) or 9"
    },
    {
        "q": "How many controls does ISO/IEC 27001:2022 have?",
        "truth": "93 controls across 4 themes (not 114 — that was the 2013 edition)",
        "trap": "Models trained on older data often say 114 (the 2013 count)"
    },
    {
        "q": "What are the six phases of TOGAF ADM in order?",
        "truth": "TOGAF ADM has 10 phases (Preliminary + A through H + Requirements Management) — not 6",
        "trap": "Models often invent a shortened 6-phase version"
    },
]

for item in conflation_questions:
    gpt_ans    = ask_gpt(
        "Answer in 1-2 sentences. Be precise with numbers.",
        item["q"], temperature=0
    )
    claude_ans = ask_claude(
        "Answer in 1-2 sentences. Be precise with numbers.",
        item["q"], temperature=0
    )

    print(f"❓ {item['q']}")
    print(f"   🟢 GPT   : {gpt_ans.strip()}")
    print(f"   🟣 Claude: {claude_ans.strip()}")
    print(f"   ✅ Truth : {item['truth']}")
    print(f"   🪤 Trap  : {item['trap']}")
    print()

print("─" * 60)
print("🔑 CONFLATION pattern: models merge details from similar frameworks.")
print("   Especially dangerous when:")
print("   • Two frameworks cover the same domain (IT governance, security)")
print("   • A framework released a new version with changed numbers")
print("   • The question sounds authoritative ('exactly', 'how many')")
print("=" * 60)

---
## 🔁 Section 6 — Self-Consistency: Detecting Unreliable Answers Automatically

**This is a production technique, not just a demo.**
For critical decisions, run the same prompt 3–5× and flag answers with low agreement.

- **High agreement ≠ correct** — but **low agreement = unreliable**
- Unambiguous tickets → 5/5 same answer → safe to auto-route
- Ambiguous tickets → 3/5 different answers → flag for human review


In [ ]:
# ── 6a. Self-consistency check ───────────────────────────────────────────────
import re
from collections import Counter

PRIORITY_SYSTEM = """
You are an IT incident prioritiser for a financial services company.
Assign exactly one priority: P1 (critical), P2 (high), P3 (medium), P4 (low).

Definitions:
P1 - Complete outage or active data loss. Revenue/regulatory impact. Act immediately.
P2 - Significant degradation OR single VIP user blocked. Act within 2 hours.
P3 - Partial impact, workaround available. Act within 8 hours.
P4 - Cosmetic or single non-critical user. Act within 3 days.

Return ONLY the priority label. No explanation.
"""

test_tickets = [
    # ── Clear anchors (should stay 100%) ──────────────────────────────────────
    "Payment gateway down. All online transactions failing. Revenue impact: £12,000/min.",
    "My mouse double-clicks when I single-click.",

    # ── Ambiguous: P1 vs P2 ───────────────────────────────────────────────────
    # Backup: data loss risk (P1) but no confirmed loss yet (P2)
    "Nightly backup finished with errors. 3 of 12 datasets show checksum mismatches. "
    "No restore has been attempted. Compliance audit is tomorrow morning.",

    # Exec login: VIP + time pressure (P1) but single user + unconfirmed (P2)
    "CFO cannot access the trading risk dashboard. Board meeting starts in 90 minutes. "
    "Issue only affects her laptop — other users on same network are fine.",

    # ── Ambiguous: P2 vs P3 ───────────────────────────────────────────────────
    # CPU: resolved but root cause unknown — could recur (P2) or one-off (P3)
    "Prod DB CPU hit 94% for 8 minutes and self-recovered. "
    "No root cause found. Same spike happened 3 weeks ago and was never explained.",

    # Email: all staff affected (P2 scope) but workaround exists (P3)
    "Email delivery delayed 20–40 mins company-wide. No mails lost. "
    "Webmail works fine. Delay started after this morning's Exchange patch.",

    # ── Ambiguous: P2 vs P4 ───────────────────────────────────────────────────
    # DR runbook: no live impact (P4) but existential risk if DR triggered tonight (P2)
    "DR failover test is scheduled for 22:00 tonight. Runbook references a load balancer "
    "decommissioned last month. Test will fail if not corrected before then.",
]

MODEL     = "gpt-4o"
RUNS      = 5
THRESHOLD = 0.8


def self_consistency(ticket: str, runs: int = RUNS) -> tuple[str, float, dict]:
    """Always uses oai_client directly — bypasses Guardrails to get vote breakdown."""
    answers = []
    for _ in range(runs):
        r = oai_client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": PRIORITY_SYSTEM},
                {"role": "user",   "content": ticket},
            ],
            temperature=0.9,
            max_tokens=10,
        )
        raw   = r.choices[0].message.content.strip().upper()  # ← uppercase HERE
        match = re.search(r'P[1-4]', raw)
        answers.append(match.group() if match else raw)

    counts    = Counter(answers)
    majority  = counts.most_common(1)[0][0]   # already uppercase
    agreement = counts.most_common(1)[0][1] / runs
    return majority, agreement, dict(counts)


# ── ALWAYS use self_consistency — Guardrails path hides vote breakdown ────────
run_fn = self_consistency
# (keep client check only if you need Guardrails for other cells)


# ── Print results ─────────────────────────────────────────────────────────────
print(f"\n{'Ticket':<54} {'Votes':<18} {'Win':>4} {'Agree':>7} {'Action':>14}")
print("─" * 104)

for ticket in test_tickets:
    majority, agreement, votes = run_fn(ticket)
    vote_str = "  ".join(f"{k}×{v}" for k, v in sorted(votes.items()))

    if agreement == 1.0:
        action = "✅ Auto-route"
    elif agreement >= THRESHOLD:
        action = "🟡 Spot-check"
    else:
        action = "🚨 Human review"

    print(f"{ticket[:53]:<54} {vote_str:<18} {majority:>4} {agreement:>7.0%} {action:>14}")

print()
print("─" * 104)
print("🔑 THRESHOLD 80%. Why these tickets should split:\n")
print("   🟡/🚨 Backup warnings   : Intact BUT unverified  → P1 risk vs P2 caution?")
print("   🟡/🚨 Exec login        : Can't replicate        → real P1 vs phantom P2?")
print("   🟡/🚨 CPU self-recovered: Spike resolved         → P2 investigate vs P3 monitor?")
print("   🟡/🚨 Email delay       : All staff affected     → P2 scope vs P3 workaround?")
print("   🟡/🚨 DR runbook        : No impact NOW          → P2 risk vs P3 schedule?")
print()
print("   If gpt-4o returns 100% on all — it is overconfident, not correct.")
print("   A human panel would disagree on every ambiguous ticket above.")
print()
print(f"   Runs: {RUNS}  |  Temp: 0.9  |  Model: {MODEL}")

In [ ]:
# ── 6b. Scored model comparison — instruction-following stress test ───────────
import re

INCIDENT_TEXT = """
At 14:32 Monday, the entire Bangalore office (approx. 300 staff) lost access to SAP.
Root cause: an expired SSL certificate on the load balancer — not flagged in the cert tool.
Engineer Arjun Menon renewed it at 15:10. Services restored by 15:18. Downtime: 46 minutes.
Financial impact estimated at ₹4.2 lakhs in lost productivity.
Certificate tool now configured to alert 30 days before expiry.
"""

STRICT_RULES = """
Summarise this IT incident report. STRICT RULES — all must be followed exactly:

1. Exactly 3 lines. No more, no less.
2. Each line must be numbered: start with "1.", "2.", "3."
3. Line 1 must begin with the word IMPACT:
4. Line 2 must begin with the word CAUSE:
5. Line 3 must begin with the word FIX:
6. Do NOT use any of these words: issue, problem, outage, failure, incident
7. Each line must be 15 words or fewer (including the prefix label) — count carefully.
8. After the 3 lines, add exactly one blank line, then write: STATUS: RESOLVED
"""

BANNED_WORDS = {"issue", "problem", "outage", "failure", "incident"}


def checks_only(scores: dict) -> dict:
    """Return only the boolean rule checks, not metadata fields."""
    skip = {"found_banned", "line_lengths", "score"}
    return {k: v for k, v in scores.items() if k not in skip}


def word_count_detail(line_lengths: list) -> str:
    parts = []
    for i, wc in enumerate(line_lengths, 1):
        if wc > 15:
            parts.append(f"L{i}:{wc}w ❌(+{wc-15})")
        else:
            parts.append(f"L{i}:{wc}w ✅")
    return " | ".join(parts)


def score_response(response: str) -> dict:
    r     = response.strip()
    lines = [l.strip() for l in r.splitlines() if l.strip()]

    content_lines = [l for l in lines if re.match(r'^[123]\.', l)]
    status_lines  = [l for l in lines if l.startswith("STATUS:")]

    rule1 = len(content_lines) == 3
    rule3 = len(content_lines) > 0 and content_lines[0].startswith("1. IMPACT:")
    rule4 = len(content_lines) > 1 and content_lines[1].startswith("2. CAUSE:")
    rule5 = len(content_lines) > 2 and content_lines[2].startswith("3. FIX:")

    found_banned = [w for w in BANNED_WORDS if w in r.lower()]
    rule6        = len(found_banned) == 0

    line_lengths = [len(l.split()) for l in content_lines]
    rule7        = all(wc <= 15 for wc in line_lengths)

    rule8 = any(l == "STATUS: RESOLVED" for l in status_lines)

    checks = {
        "3 numbered lines" : rule1,
        "IMPACT: prefix"   : rule3,
        "CAUSE: prefix"    : rule4,
        "FIX: prefix"      : rule5,
        "no banned words"  : rule6,
        "≤15 words/line"   : rule7,
        "STATUS: RESOLVED" : rule8,
    }

    return {
        **checks,
        "found_banned" : found_banned,
        "line_lengths" : line_lengths,
        "score"        : sum(checks.values()) / len(checks),
    }


MODELS = [
    ("gpt-4o-mini",     "GPT-4o-mini",  lambda p: ask_gpt(p,    INCIDENT_TEXT, temperature=0)),
    ("claude-haiku-4-5","Claude Haiku", lambda p: ask_claude(p, INCIDENT_TEXT, temperature=0)),
]

print("=" * 64)
print("6b. INSTRUCTION-FOLLOWING STRESS TEST")
print("=" * 64)

results = {}
for model_id, label, call_fn in MODELS:
    response       = call_fn(STRICT_RULES)
    scores         = score_response(response)
    results[label] = scores

    pct   = scores["score"]
    grade = "🏆" if pct == 1.0 else ("✅" if pct >= 0.8 else "⚠️ ")

    print(f"\n── {grade} {label}  ({pct:.0%}) ──")
    for line in response.strip().splitlines():
        print(f"   {line}")
    print()

    for rule, passed in checks_only(scores).items():
        icon  = "✅" if passed else "❌"
        extra = ""
        if rule == "≤15 words/line":
            extra = f"   {word_count_detail(scores['line_lengths'])}"
        if rule == "no banned words" and scores["found_banned"]:
            extra = f"   (found: {scores['found_banned']})"
        print(f"   {icon} {rule}{extra}")

# ── Head-to-head table ────────────────────────────────────────────────────────
print()
print("─" * 64)
print(f"{'Rule':<22} {'GPT-4o-mini':>16} {'Claude Haiku':>16}")
print("─" * 64)

rule_keys = [
    "3 numbered lines", "IMPACT: prefix", "CAUSE: prefix",
    "FIX: prefix", "no banned words", "≤15 words/line", "STATUS: RESOLVED"
]
for rule in rule_keys:
    g = "✅" if results["GPT-4o-mini"][rule]  else "❌"
    c = "✅" if results["Claude Haiku"][rule] else "❌"
    print(f"{rule:<22} {g:>16} {c:>16}")

print("─" * 64)
print(f"{'TOTAL SCORE':<22} {results['GPT-4o-mini']['score']:>15.0%} {results['Claude Haiku']['score']:>15.0%}")

# ── Workshop punchline ────────────────────────────────────────────────────────
print()
print("─" * 64)
print("🔑 KEY FINDING:\n")
print("   Both models broke the SAME rule — ≤15 words per line.")
print("   They overshot by just 1–2 words. Not laziness — a fundamental")
print("   LLM behaviour: when format conflicts with completeness,")
print("   the model silently prioritises completeness.\n")
print("   ┌──────────────────────────────────────────────────────┐")
print("   │  Prompt rules are suggestions. Not constraints.     │")
print("   │  The model 'knows' 16 > 15. It just didn't care.   │")
print("   └──────────────────────────────────────────────────────┘")
print()
print("   Week 2 → JSON mode + Pydantic max_length enforces this")
print("   structurally. Hard constraint, not a suggestion.")

---
## 🔍 Section 7 — Fix Stale Answers with Tavily Live Search

**The problem:** LLMs have a training cutoff. Anything that changed after that date will be wrong.

**The fix:** Before answering, search the web for fresh data and inject it into the prompt.
This is called **Retrieval-Augmented Generation (RAG)** — we'll do a full deep-dive in Week 3.

> 🆓 **Free tier:** 1,000 searches/month at [tavily.com](https://tavily.com) — no credit card needed.

```
WITHOUT TAVILY:                 WITH TAVILY:
User asks question              User asks question
       ↓                               ↓
LLM answers from                Tavily searches web → gets fresh results
training data                          ↓
(may be outdated)               Fresh results injected into prompt
                                       ↓
                                LLM answers from fresh data
                                (grounded, accurate)
```


In [ ]:
# ── Tavily setup ──────────────────────────────────────────────────────────────
from tavily import TavilyClient

if not TAVILY_API_KEY:
    print("❌ TAVILY_API_KEY not found in .env")
    print("   Get a free key at: https://tavily.com")
    print("   Add to .env: TAVILY_API_KEY=tvly-...")
else:
    tavily = TavilyClient(api_key=TAVILY_API_KEY)
    print("✅ Tavily client ready")
    print()
    results = tavily.search("latest Python version", max_results=2)
    for r in results["results"]:
        print(f"  📰 {r['title']}")
        print(f"     {r['content'][:120]}...")
        print()


In [ ]:
# ── Side-by-side: LLM alone vs LLM + Tavily ──────────────────────────────────
def search_and_ask(question, search_query=None):
    query    = search_query or question
    results  = tavily.search(query, max_results=3)
    snippets = "\n\n".join([
        f"Source: {r['url']}\n{r['content'][:300]}"
        for r in results["results"]
    ])
    augmented = f"""Use ONLY the search results below to answer the question.
If the results don't contain the answer, say so. Cite the source URL.

SEARCH RESULTS:
{snippets}

QUESTION: {question}"""
    return ask_gpt("You are a factual assistant. Use only the provided search results.",
               augmented, temperature=0), snippets

if TAVILY_API_KEY:
    print("=" * 60)
    print("LLM ALONE  vs  LLM + TAVILY LIVE SEARCH")
    print("=" * 60)
    print()

    questions_to_fix = [
        ("What is the latest stable version of Python?", "Python latest stable release"),
        ("Who is the current CEO of Wipro?",            "Wipro CEO current 2026"),
    ]

    for question, search_query in questions_to_fix:
        print(f"❓ {question}")
        print()
        llm_only = ask_gpt("Answer directly.", question, temperature=0)
        print(f"   ❌ LLM alone (may be outdated):")
        print(f"      {llm_only[:150]}")
        print()
        llm_grounded, _ = search_and_ask(question, search_query)
        print(f"   ✅ LLM + Tavily (fresh from the web):")
        for line in llm_grounded.splitlines():
            print(f"      {line}")
        print()
        print("   " + "─" * 50)
        print()
else:
    print("⚠️  Skipping Tavily demo — add TAVILY_API_KEY to your .env file")
    print("   Get a free key at https://tavily.com")


In [ ]:
# ── IT-specific live search: vulnerability check ──────────────────────────────
if TAVILY_API_KEY:
    print("=" * 60)
    print("PRACTICAL IT USE CASE — Live vulnerability check")
    print("=" * 60)
    print()
    vuln_question = "Are there any known critical vulnerabilities in Windows 11 23H2 released in 2024?"
    print(f"Question: {vuln_question}")
    print()
    vuln_answer, _ = search_and_ask(
        vuln_question, "Windows 11 23H2 critical vulnerabilities CVE 2024"
    )
    print("📋 Grounded answer:")
    print(vuln_answer)
    print()
    print("💡 Without Tavily, the model would miss vulnerabilities discovered after its cutoff.")
else:
    print("⚠️  Add TAVILY_API_KEY  to your .env to run this demo")


---
## 🎛️ Section 8 — Build Your Own: Interactive Prompt Lab

Now it's your turn. Modify the cells below to experiment.


In [ ]:
# ── Exercise A: Change the persona ───────────────────────────────────────────
# Ideas: "pirate", "Yoda", "grumpy senior developer", "5-year-old"

MY_SYSTEM_PROMPT = """
You are a friendly IT assistant who explains everything using cooking analogies.
Keep it simple and fun.
"""
MY_QUESTION = "What is a firewall and why do we need one?"

gpt_response = ask_gpt(MY_SYSTEM_PROMPT, MY_QUESTION, temperature=0.7)
print("🟢 GPT-4o-mini says:")
print(gpt_response)
print()

claude_response = ask_claude(MY_SYSTEM_PROMPT, MY_QUESTION, temperature=0.7)
print("🟣 Claude Haiku says:")
print(claude_response)


In [ ]:
# ── Exercise B: Format control ────────────────────────────────────────────────
FORMAT_SYSTEM = """
You are an IT incident analyst.
Always respond in this exact format:

SEVERITY: [P1/P2/P3/P4]
CATEGORY: [one word]
AFFECTED: [number of users]
ACTION:   [one sentence — what to do right now]
"""

incidents_to_classify = [
    "The payment gateway has been returning errors for 20 minutes. 300+ customers affected.",
    "My mouse cursor sometimes jumps across the screen.",
    "The entire Bangalore office Wi-Fi is down. 500 staff cannot work.",
]

print("=" * 60)
print("STRUCTURED OUTPUT DEMO")
print("=" * 60)

for incident in incidents_to_classify:
    response = ask_gpt(FORMAT_SYSTEM, incident, temperature=0)
    label = (incident[:67] + "...") if len(incident) > 70 else incident
    print(f"\nIncident: {label}")
    print(response)
    print()


In [ ]:
# ── 8a. ChatSession — conversation history manager ───────────────────────────
class ChatSession:
    """
    Manages conversation history for a multi-turn chatbot.
    Implements a sliding window to keep context within token limits.
    """
    def __init__(self, system: str, model: str = "gpt-4o-mini",
                 max_history_turns: int = 10, temperature: float = 0.3):
        self.system            = system
        self.model             = model
        self.max_history_turns = max_history_turns
        self.temperature       = temperature
        self.history: list     = []
        self.turn_count        = 0

    def chat(self, user_message: str):
        self.turn_count += 1
        self.history.append({"role": "user", "content": user_message})
        window = self.history[-(self.max_history_turns * 2):]
        messages = [{"role": "system", "content": self.system}, *window]
        resp = oai_client.chat.completions.create(
            model=self.model, messages=messages,
            temperature=self.temperature, max_tokens=400
        )
        response = resp.choices[0].message.content
        ctx_tokens = sum(len(m["content"].split()) * 1.3 for m in messages)
        self.history.append({"role": "assistant", "content": response})
        return response, int(ctx_tokens)

    def show_history(self) -> None:
        print(f"\n{'─'*60}  Conversation history ({self.turn_count} turns)")
        for msg in self.history:
            role = "👤 User" if msg["role"] == "user" else "🤖 Bot "
            print(f"  {role}: {msg['content'][:120]}")

    def reset(self) -> None:
        self.history = []
        self.turn_count = 0


HELPDESK_SYSTEM = """
You are an IT helpdesk assistant for a technology company in Bangalore.
Help employees with IT issues: software, hardware, network, access, and data problems.
Keep responses concise and actionable. Ask one clarifying question at a time if needed.
If the issue sounds like a P1, say so clearly and tell the user to also call the helpdesk line.
You do not have access to real systems — you can guide users through steps, not take actions.
"""

bot = ChatSession(system=HELPDESK_SYSTEM, max_history_turns=8, temperature=0.3)

conversation = [
    "Hi, my Outlook is not receiving new emails since this morning.",
    "Yes, I can send emails but nothing is arriving. My colleagues say they sent me things.",
    "I'm on Outlook 2021, Windows 11. I restarted already.",
    "Also, while we're at it — my VPN drops every 30 minutes too.",
]

print("Simulating a 4-turn helpdesk conversation...\n")
for user_msg in conversation:
    print(f"👤 User : {user_msg}")
    response, ctx_tokens = bot.chat(user_msg)
    print(f"🤖 Bot  : {response.strip()[:300]}")
    print(f"   [context: ~{ctx_tokens} tokens]")
    print()

bot.show_history()


In [ ]:
# ── 8b. Common mistake — stateless calls (no history) ────────────────────────
print("=" * 60)
print("COMMON MISTAKE — Stateless calls (no history)")
print("=" * 60)
print()
print("Each call knows nothing about previous turns:")
print()

turns_stateless = [
    "My Outlook is not receiving emails.",
    "Yes, I already told you — I can send but not receive.",
    "As I said, I restarted twice.",
]

for turn in turns_stateless:
    print(f"👤 User : {turn}")
    resp = ask_gpt(HELPDESK_SYSTEM, turn, temperature=0.3)
    print(f"🤖 Bot  : {resp.strip()[:200]}")
    print()

print("🔑 KEY LESSON: The bot repeats itself, asks questions the user already answered,")
print("   and loses all diagnostic progress.")
print("   ALWAYS pass full conversation history — not just the last message.")


---
## 🚀 Bonus — Streamlit Chatbot (run as a web app)

Save the cell below as `chatbot_w01.py` and run:
```bash
streamlit run chatbot_w01.py
```


In [ ]:
# ── Save Streamlit chatbot to file ─────────────────────────────────────────
# Run: streamlit run chatbot_w01.py
import textwrap
_lines = ['import os\n', 'import streamlit as st\n', 'from openai import OpenAI\n', '\n', 'SYSTEM = "You are an IT helpdesk assistant. Help employees with IT issues. Keep responses concise. Ask one clarifying question at a time."\n', '\n', 'st.set_page_config(page_title="IT Helpdesk Bot -- W1", page_icon="🖥️")\n', 'st.title("🖥️  IT Helpdesk Chatbot")\n', 'st.caption("Week 1 -- multi-turn chatbot with token tracking")\n', '\n', 'if "history" not in st.session_state:\n', '    st.session_state.history = []\n', 'if "total_tokens" not in st.session_state:\n', '    st.session_state.total_tokens = 0\n', '\n', 'client = OpenAI()\n', '\n', 'with st.sidebar:\n', '    st.metric("Turns", len(st.session_state.history) // 2)\n', '    st.metric("~Total tokens", st.session_state.total_tokens)\n', '    if st.button("Reset conversation"):\n', '        st.session_state.history = []\n', '        st.session_state.total_tokens = 0\n', '        st.rerun()\n', '\n', 'for msg in st.session_state.history:\n', '    with st.chat_message(msg["role"]):\n', '        st.write(msg["content"])\n', '\n', 'if prompt := st.chat_input("Describe your IT issue..."):\n', '    st.session_state.history.append({"role": "user", "content": prompt})\n', '    with st.chat_message("user"):\n', '        st.write(prompt)\n', '    messages = [{"role": "system", "content": SYSTEM}, *st.session_state.history[-16:]]\n', '    with st.chat_message("assistant"):\n', '        with st.spinner("Thinking..."):\n', '            resp = client.chat.completions.create(\n', '                model="gpt-4o-mini", messages=messages, temperature=0.3, max_tokens=400\n', '            )\n', '            response = resp.choices[0].message.content\n', '            st.session_state.total_tokens += (resp.usage.total_tokens if resp.usage else 0)\n', '        st.write(response)\n', '    st.session_state.history.append({"role": "assistant", "content": response})\n', '    st.rerun()\n']
with open('chatbot_w01.py', 'w') as f:
    f.writelines(_lines)
print('✅ Streamlit chatbot saved to chatbot_w01.py')
print('Run: streamlit run chatbot_w01.py')


---
## 📊 Section 10 — Session Summary

In [ ]:
# ── Session cost summary ─────────────────────────────────────────────────────
import tempfile

if client is not None:
    banner("Session tracer — LLMClient calls only")
    print("  Most of this notebook used raw SDK clients on purpose.")
    print("  Only calls through LLMClient appear below (cost calc, self-consistency, etc.).\n")
    tracer.summary()
    trace_path = os.path.join(tempfile.gettempdir(), "w01_traces.jsonl")
    tracer.export_jsonl(trace_path)
    print(f"\nTraces exported to {trace_path}")
else:
    estimates = [
        ("Token visualiser (Section 1)",         250,  80),
        ("Character counting (Section 1)",        350, 150),
        ("Next-word probs (Section 2)",           200,  15),
        ("Temperature experiments (Section 2)",   900, 250),
        ("System prompt personas (Section 3)",    600, 450),
        ("GPT vs Claude comparison (Section 4)", 1200, 600),
        ("Hallucination demos (Section 5)",       900, 450),
        ("Self-consistency (Section 6)",          600, 100),
        ("Tavily + grounded answers (Section 7)", 900, 300),
        ("Chatbot demo (Section 9)",              800, 600),
    ]
    total_in  = sum(e[1] for e in estimates)
    total_out = sum(e[2] for e in estimates)
    est_cost  = (total_in * 0.00015 + total_out * 0.00060) / 1000
    print(f"  {'Section':<45} {'~Input':>8} {'~Output':>8}")
    print("  " + "─" * 65)
    for name, inp, out in estimates:
        print(f"  {name:<45} {inp:>8,} {out:>8,}")
    print("  " + "─" * 65)
    print(f"  {'TOTAL':<45} {total_in:>8,} {total_out:>8,}")
    print()
    print(f"  💰 Estimated cost: ~${est_cost:.5f} USD  (fraction of a rupee)")
    print()
    print("  Week 2 switches to LLMClient for every call — full tracing from the start.")


---
## ✅ Week 1 — Key Takeaways

| What we learned | Why it matters | Production rule |
|---|---|---|
| Models read **tokens**, not words | Hindi = 3× more tokens = more cost | Budget by token, not word |
| LLMs **predict the next token** probabilistically | Sophisticated autocomplete, not databases | Never use for exact counting/parsing |
| **Temperature** controls randomness | temp=0 for decisions, high for brainstorming | Never use temp > 0.5 in routing pipelines |
| **System prompt** shapes everything | Role, tone, format, rules | Always define a system prompt in production |
| **GPT and Claude** differ | Same question → different answers | Agreement ≠ truth; verify critical facts |
| Hallucination has **3 patterns** | Fabrication, stale data, conflation | Every LLM fact needs a verification strategy |
| **Self-consistency** flags unreliable answers | Low agreement = model is uncertain | Use n=3 check before auto-routing high-stakes decisions |
| **Tavily** gives models live data | Fixes training cutoff | Foundation of RAG (Week 3 deep-dive) |
| **Conversation history** is essential | Stateless calls lose context | Always pass full history — not just the last message |

---

### 🚀 Coming up in Week 2
We'll learn to **control the model's output reliably**:
- Why naive prompts break in production
- Few-shot examples that teach business logic
- JSON output that your systems can parse
- Prompt injection attacks — and how to defend against them

---

### 📝 Before next week
1. Pick a real question from your work domain and ask it to both GPT and Claude
2. If it's a fast-changing fact, use the Tavily cell (Exercise C) to get a live answer
3. Run the Streamlit chatbot and share it with a colleague
4. Note: did the model admit uncertainty, or sound confident regardless?
